In [0]:
import time
from datetime import datetime, timezone

import pandas as pd
import requests

In [0]:
indicators = [
    "NCDMORT3070", "SDGPM25", "WHOSIS_000001", "WHOSIS_000002", "MDG_0000000029", "HIV_0000000026", "NCD_GLUC_04", "BP_04", "NCD_BMI_30C", "M_Est_smk_curr_std", "WSH_WATER_SAFELY_MANAGED", "WSH_SANITATION_SAFELY_MANAGED", "NUTRITION_ANT_HAZ_NE2", "NUTRITION_ANAEMIA_REPRODUCTIVEAGE_PREV", "GHED_GGHE-D_pc_US_SHA2011",
]

In [0]:
base_url = "https://ghoapi.azureedge.net/api/"
dfs = []
errors = []

for code in indicators:
    try:
        response = requests.get(f"{base_url}{code}", timeout=30)
        response.raise_for_status()
        values = response.json().get("value")
        if not values:
            raise ValueError("empty response")
        dfs.append(pd.DataFrame(values))
        print(f"{code}: {len(values)} rows")
    except Exception as e:
        print(f"{code}: error: {e}")
        errors.append(code)
    time.sleep(2)

assert not errors, f"Ingestion failed for indicators: {errors}"

df_pandas = pd.concat(dfs, ignore_index=True)
print(f"Total: {df_pandas.shape}")

In [0]:
df_pandas["_ingested_at"] = datetime.now(timezone.utc).isoformat()
df_pandas["_source_url"] = base_url

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

df_spark = spark.createDataFrame(df_pandas.astype(str).where(df_pandas.notna(), None))

(df_spark.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.gho_raw"))